# KrushikaDhara ONNX to TFLite Conversion (Robust Validation)

Upload `best_model.onnx` and `dataset_split/test/` to the Colab environment before running.

In [ ]:
!pip install onnx onnx-tf tensorflow torch torchvision torchaudio


In [ ]:
import sys
import onnx
import tensorflow as tf
from onnx_tf.backend import prepare
import numpy as np
import glob
from PIL import Image
import json

print("===============================")
print("Environment Versions")
print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"ONNX Version: {onnx.__version__}")
print("===============================")


In [ ]:
onnx_path = "best_model.onnx"
tf_model_path = "saved_model"
tflite_path = "crop_disease_classifier_int8.tflite"
test_dir = "test/"

print("Loading ONNX model...")
onnx_model = onnx.load(onnx_path)
print("ONNX Input/Output validation...")

input_shape = []
for inp in onnx_model.graph.input:
    input_shape = [dim.dim_value for dim in inp.type.tensor_type.shape.dim]
    print(f"ONNX Input: {inp.name}, Shape: {input_shape}")
    
output_shape = []
for out in onnx_model.graph.output:
    output_shape = [dim.dim_value for dim in out.type.tensor_type.shape.dim]
    print(f"ONNX Output: {out.name}, Shape: {output_shape}")


In [ ]:
print("Converting ONNX to TensorFlow SavedModel...")
tf_rep = prepare(onnx_model)
tf_rep.export_graph(tf_model_path)
print("SavedModel export successful.")


In [ ]:
print("Converting to INT8 TFLite with Calibration...")

def representative_dataset():
    image_paths = glob.glob(f"{test_dir}/*/*.*")
    np.random.shuffle(image_paths)
    count = 0
    for img_path in image_paths:
        if count >= 100: break
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((224, 224))
            # Training Contract: 
            # ImageNet mean=[0.485, 0.456, 0.406] std=[0.229, 0.224, 0.225]
            img_array = np.array(img, dtype=np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_array = (img_array - mean) / std
            img_array = np.transpose(img_array, (2, 0, 1)) # NCHW
            img_array = np.expand_dims(img_array, axis=0)
            count += 1
            yield [img_array]
        except Exception as e:
            print(f"Skipping image due to error: {e}")

converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print("TFLite INT8 conversion successful!")


In [ ]:
print("TFLite Metadata Verification...")
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

metadata = {
    "input_shape": input_details["shape"].tolist(),
    "input_dtype": str(input_details["dtype"]),
    "input_scale": float(input_details["quantization"][0]),
    "input_zero_point": int(input_details["quantization"][1]),
    "output_shape": output_details["shape"].tolist(),
    "output_dtype": str(output_details["dtype"]),
    "output_scale": float(output_details["quantization"][0]),
    "output_zero_point": int(output_details["quantization"][1])
}

print(json.dumps(metadata, indent=4))
with open("tflite_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("Conversion pipeline complete. Download crop_disease_classifier_int8.tflite and tflite_metadata.json")
